In [ ]:
import os
import glob
import random
import re

from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as transforms
from torchvision.utils import save_image
from torchvision import models

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

import sys
sys.path.append('/kaggle/working')

In [ ]:
def set_seed(seed=712):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(712)

In [ ]:
class PartialConv2d(nn.Conv2d):
    def __init__(self, *args, **kwargs):
        if 'multi_channel' in kwargs:
            self.multi_channel = kwargs.pop('multi_channel')
        else:
            self.multi_channel = False

        if 'return_mask' in kwargs:
            self.return_mask = kwargs.pop('return_mask')
        else:
            self.return_mask = False

        super(PartialConv2d, self).__init__(*args, **kwargs)

        if self.multi_channel:
            self.weight_maskUpdater = torch.ones(self.out_channels, self.in_channels, self.kernel_size[0], self.kernel_size[1])
        else:
            self.weight_maskUpdater = torch.ones(1, 1, self.kernel_size[0], self.kernel_size[1])

        self.slide_winsize = self.weight_maskUpdater.shape[1] * self.weight_maskUpdater.shape[2] * self.weight_maskUpdater.shape[3]
        self.last_size = (None, None, None, None)
        self.update_mask = None
        self.mask_ratio = None

    def forward(self, input, mask_in=None):
        assert len(input.shape) == 4
        if mask_in is not None or self.last_size != tuple(input.shape):
            self.last_size = tuple(input.shape)
            with torch.no_grad():
                if self.weight_maskUpdater.type() != input.type():
                    self.weight_maskUpdater = self.weight_maskUpdater.to(input)
                mask = mask_in if mask_in is not None else torch.ones(1, 1, input.data.shape[2], input.data.shape[3]).to(input)
                self.update_mask = F.conv2d(mask, self.weight_maskUpdater, bias=None, stride=self.stride, padding=self.padding, dilation=self.dilation, groups=1)
                self.mask_ratio = self.slide_winsize / (self.update_mask + 1e-8)
                self.update_mask = torch.clamp(self.update_mask, 0, 1)
                self.mask_ratio = torch.mul(self.mask_ratio, self.update_mask)

        raw_out = super(PartialConv2d, self).forward(torch.mul(input, mask_in) if mask_in is not None else input)

        if self.bias is not None:
            bias_view = self.bias.view(1, self.out_channels, 1, 1)
            output = torch.mul(raw_out - bias_view, self.mask_ratio) + bias_view
            output = torch.mul(output, self.update_mask)
        else:
            output = torch.mul(raw_out, self.mask_ratio)

        if self.return_mask:
            return output, self.update_mask
        return output

In [ ]:
def gram_matrix(x):
    (b, ch, h, w) = x.size()
    features = x.view(b, ch, w * h)
    gram = torch.baddbmm(torch.zeros(b, ch, ch).to(x.device), features, features.transpose(1, 2), beta=0, alpha=1./(ch * h * w))
    return gram

class InpaintingLoss(nn.Module):
    def __init__(self, vgg_device):
        super().__init__()
        vgg16 = models.vgg16(pretrained=True).features.to(vgg_device).eval()
        self.enc_vgg = nn.ModuleList([vgg16[:4], vgg16[4:9], vgg16[9:16]])
        for p in self.parameters(): p.requires_grad = False
        self.l1 = nn.L1Loss()

    def forward(self, mask, output, gt):
        comp = mask * gt + (1 - mask) * output
        # L1 Losses
        l_valid = self.l1(mask * output, mask * gt)
        l_hole = self.l1((1 - mask) * output, (1 - mask) * gt)

        # VGG Losses
        out_feat = self.get_vgg_feat(output)
        gt_feat = self.get_vgg_feat(gt)
        comp_feat = self.get_vgg_feat(comp)

        l_perc = sum(self.l1(o, g) + self.l1(c, g) for o, g, c in zip(out_feat, gt_feat, comp_feat))
        l_style = sum(self.l1(gram_matrix(o), gram_matrix(g)) + self.l1(gram_matrix(c), gram_matrix(g)) for o, g, c in zip(out_feat, gt_feat, comp_feat))

        # TV Loss
        l_tv = torch.mean(torch.abs(comp[:, :, :, :-1] - comp[:, :, :, 1:])) + torch.mean(torch.abs(comp[:, :, :-1, :] - comp[:, :, 1:, :]))

        return l_valid + 6*l_hole + 0.05*l_perc + 120*l_style + 0.1*l_tv

    def get_vgg_feat(self, x):
        feats = []
        for layer in self.enc_vgg:
            x = layer(x)
            feats.append(x)
        return feats

In [ ]:
class PConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, k, s, p, use_bn=True, act='relu'):
        super().__init__()
        self.pconv = PartialConv2d(in_ch, out_ch, k, s, p, multi_channel=True, return_mask=True)
        self.bn = nn.BatchNorm2d(out_ch) if use_bn else None
        self.act = nn.ReLU(inplace=True) if act=='relu' else nn.LeakyReLU(0.2, inplace=True) if act=='leaky' else None

    def forward(self, x, m):
        x, m = self.pconv(x, m)
        if self.bn: x = self.bn(x)
        if self.act: x = self.act(x)
        return x, m

class PConvUNet(nn.Module):
    def __init__(self):
        super().__init__()
        # Encoder
        self.enc1 = PConvBlock(3, 64, 7, 2, 3, use_bn=False)
        self.enc2 = PConvBlock(64, 128, 5, 2, 2)
        self.enc3 = PConvBlock(128, 256, 5, 2, 2)
        self.enc4 = PConvBlock(256, 512, 3, 2, 1)
        for i in range(5, 9): setattr(self, f'enc{i}', PConvBlock(512, 512, 3, 2, 1))

        # Decoder
        for i in range(9, 13): setattr(self, f'dec{i}', PConvBlock(512+512, 512, 3, 1, 1, act='leaky'))
        self.dec13 = PConvBlock(512+256, 256, 3, 1, 1, act='leaky')
        self.dec14 = PConvBlock(256+128, 128, 3, 1, 1, act='leaky')
        self.dec15 = PConvBlock(128+64, 64, 3, 1, 1, act='leaky')
        self.dec16 = PConvBlock(64+3, 3, 3, 1, 1, use_bn=False, act='none')

    def forward(self, x, m):
        e1, m1 = self.enc1(x, m)
        e2, m2 = self.enc2(e1, m1)
        e3, m3 = self.enc3(e2, m2)
        e4, m4 = self.enc4(e3, m3)
        e5, m5 = self.enc5(e4, m4)
        e6, m6 = self.enc6(e5, m5)
        e7, m7 = self.enc7(e6, m6)
        e8, m8 = self.enc8(e7, m7)

        def up(x, m, e, em):
            x, m = F.interpolate(x, scale_factor=2), F.interpolate(m, scale_factor=2)
            return torch.cat([x, e], 1), torch.cat([m, em], 1)

        d, dm = self.dec9(*up(e8, m8, e7, m7))
        d, dm = self.dec10(*up(d, dm, e6, m6))
        d, dm = self.dec11(*up(d, dm, e5, m5))
        d, dm = self.dec12(*up(d, dm, e4, m4))
        d, dm = self.dec13(*up(d, dm, e3, m3))
        d, dm = self.dec14(*up(d, dm, e2, m2))
        d, dm = self.dec15(*up(d, dm, e1, m1))
        out, _ = self.dec16(*up(d, dm, x, m))
        return out

In [ ]:
# =========================================
# 1. CẤU HÌNH ĐƯỜNG DẪN TẬP TEST
# =========================================ị
TEST_IMG_DIR = '../data/test_dataset' 

TEST_MASK_DIR = '../data/test_mask/testing_mask_dataset'

MODEL_WEIGHT = '../checkpoint/pconv_celeba_epoch_10.pth'

# Nơi lưu các bức ảnh kết quả sau khi vẽ xong
SAVE_RESULT_DIR = '../my_results'
os.makedirs(SAVE_RESULT_DIR, exist_ok=True)

# =========================================
# 2. KHỞI TẠO VÀ NẠP MÔ HÌNH
# =========================================
print("Đang nạp não bộ mô hình...")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
net = PConvUNet().to(device)

if torch.cuda.device_count() > 1:
    net = nn.DataParallel(net)

checkpoint = torch.load(MODEL_WEIGHT, map_location=device)
state_dict = checkpoint['model_state_dict'] if 'model_state_dict' in checkpoint else checkpoint

if isinstance(net, nn.DataParallel) and not list(state_dict.keys())[0].startswith('module.'):
    state_dict = {f'module.{k}': v for k, v in state_dict.items()}
elif not isinstance(net, nn.DataParallel) and list(state_dict.keys())[0].startswith('module.'):
    state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}

net.load_state_dict(state_dict)
net.eval()
print("Nạp mô hình thành công!\n")

# =========================================
# 3. CHẠY KIỂM THỬ TRÊN TẬP ẢNH CỦA BẠN
# =========================================
# Lấy danh sách ảnh và mask
img_paths = glob.glob(os.path.join(TEST_IMG_DIR, '*.*'))
if len(img_paths) == 0:
    raise ValueError("Không tìm thấy ảnh nào trong TEST_IMG_DIR!")

img_paths_10 = random.sample(img_paths, 10) # Lấy 10 ảnh để test

mask_paths = glob.glob(os.path.join(TEST_MASK_DIR, '*.*'))




transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor()
])

def tensor_to_img(tensor):
    img = tensor.squeeze(0).cpu().numpy()
    img = np.transpose(img, (1, 2, 0))
    img = np.clip(img, 0, 1)
    return img

print(f"Bắt đầu vẽ lại {len(img_paths_10)} bức ảnh...\n")

with torch.no_grad():
    for i, img_path in enumerate(img_paths_10):
        # 1. Đọc và tiền xử lý ảnh
        raw_img = Image.open(img_path).convert('RGB')
        img_tensor = transform(raw_img).unsqueeze(0).to(device)
        
        # 2. Đọc Mask
        mask_path = random.choice(mask_paths)
        raw_mask = Image.open(mask_path).convert('L')
        mask_tensor = transform(raw_mask)
        mean_val = mask_tensor.mean().item()
        if mean_val > 0.5:
            # Nếu trắng chiếm đa số -> Nền trắng, Rách đen
            mask_tensor = (mask_tensor > 0.5).float()
        else:
            # Nếu đen chiếm đa số -> Nền đen, Rách trắng -> Cần lật lại
            mask_tensor = (mask_tensor < 0.5).float()
        mask_tensor = mask_tensor.repeat(3, 1, 1).unsqueeze(0).to(device)
        
        # 3. Đưa qua mô hình
        input_img = img_tensor * mask_tensor # Ảnh rách
        output_tensor = net(input_img, mask_tensor) # Mô hình dự đoán
        
        # Cắt ghép: Nền thật + Nét vẽ của mô hình
        final_result = mask_tensor * img_tensor + (1 - mask_tensor) * output_tensor
        
        # 4. Trình bày kết quả bằng Matplotlib
        plt.figure(figsize=(12, 4))
        
        plt.subplot(1, 3, 1)
        plt.title(f"Ảnh gốc {i+1}")
        plt.imshow(tensor_to_img(img_tensor))
        plt.axis('off')

        plt.subplot(1, 3, 2)
        plt.title("Bị rách")
        plt.imshow(tensor_to_img(input_img))
        plt.axis('off')

        
        plt.subplot(1, 3, 3)
        plt.title("Mô hình vẽ lại")
        plt.imshow(tensor_to_img(final_result))
        plt.axis('off')
        
        # Lưu kết quả xuống ổ đĩa để tải về đưa vào báo cáo
        save_path = os.path.join(SAVE_RESULT_DIR, f"result_{i+1}.png")
        plt.savefig(save_path, bbox_inches='tight', pad_inches=0.1)
        
        # Hiển thị trực tiếp trên notebook
        plt.show()
        plt.close()

print(f"\nToàn bộ kết quả đã được lưu tại: {SAVE_RESULT_DIR}")

In [ ]:
from torchmetrics.image import PeakSignalNoiseRatio, StructuralSimilarityIndexMeasure

psnr_metric = PeakSignalNoiseRatio(data_range=1.0).to(device)
ssim_metric = StructuralSimilarityIndexMeasure(data_range=1.0).to(device)

print(f"Bắt đầu đánh giá trên tập TEST...\n")
net.eval()

all_l1_losses = []
all_psnr_scores = []
all_ssim_scores = []

class TrainDataset(Dataset):
    def __init__(self, image_paths, mask_dir, img_size=256):
        self.image_paths = image_paths
        self.mask_paths = glob.glob(os.path.join(mask_dir, '*.*'))

        self.img_transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor()
        ])
        self.mask_transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        gt_img = Image.open(self.image_paths[idx]).convert('RGB')
        gt_img = self.img_transform(gt_img)

        mask_path = random.choice(self.mask_paths)
        mask = Image.open(mask_path).convert('L')
        mask = self.mask_transform(mask)

        # AUTO-FIX: Tự động chuẩn hóa Mask (Nền = 1, Rách = 0)
        mean_val = mask.mean().item()
        if mean_val > 0.5:
            # Nếu trắng chiếm đa số -> Nền trắng, Rách đen
            mask = (mask > 0.5).float()
        else:
            # Nếu đen chiếm đa số -> Nền đen, Rách trắng -> Cần lật lại
            mask = (mask < 0.5).float()

        mask = mask.repeat(3, 1, 1)
        
        return gt_img, mask

test_dataset = TrainDataset(img_paths, TEST_MASK_DIR, img_size=256)

test_loader = DataLoader(
    test_dataset, 
    batch_size=16,
    shuffle=False, 
    num_workers=2, 
    pin_memory=True
)

with torch.no_grad():
    for gt_imgs, masks in tqdm(test_loader, desc="Đang chấm điểm"):
        gt_imgs = gt_imgs.to(device)
        masks = masks.to(device)
        
        input_imgs = gt_imgs * masks
        
        outputs = net(input_imgs, masks)
        
        comp_imgs = masks * gt_imgs + (1 - masks) * outputs
        
        # Tính toán các chỉ số cho batch hiện tại
        l1 = F.l1_loss(comp_imgs, gt_imgs).item()
        psnr = psnr_metric(comp_imgs, gt_imgs).item()
        ssim = ssim_metric(comp_imgs, gt_imgs).item()
        
        all_l1_losses.append(l1)
        all_psnr_scores.append(psnr)
        all_ssim_scores.append(ssim)

# Tính điểm số trung bình toàn tập Test
avg_l1 = np.mean(all_l1_losses)
avg_psnr = np.mean(all_psnr_scores)
avg_ssim = np.mean(all_ssim_scores)

print("\n" + "="*50)
print("KẾT QUẢ TRUNG BÌNH TRÊN TẬP TEST")
print(f"🔹 L1 Loss (Càng thấp càng tốt): {avg_l1:.4f}")
print(f"🔹 PSNR    (Càng cao càng tốt): {avg_psnr:.2f} dB")
print(f"🔹 SSIM    (Càng gần 1 càng tốt): {avg_ssim:.4f}")
print("="*50 + "\n")

In [ ]:
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('HIỆU SUẤT MÔ HÌNH TRÊN TẬP TEST', fontsize=16, fontweight='bold', y=1.05)

# Subplot 1: Khảo sát L1 Loss và SSIM trung bình
metrics_names = ['L1 Loss', 'SSIM']
metrics_values = [avg_l1, avg_ssim]
bars = axes[0].bar(metrics_names, metrics_values, color=['#e74c3c', '#2ecc71'], width=0.4)
axes[0].set_title('Chỉ số L1 và SSIM Trung bình', fontsize=12, pad=10)
axes[0].set_ylim(0, max(metrics_values) + 0.1)
for bar in bars:
    yval = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2, yval + 0.01, f'{yval:.4f}', ha='center', va='bottom', fontweight='bold')

# Subplot 2: Phân phối PSNR
sns.histplot(all_psnr_scores, bins=20, kde=True, color='#3498db', ax=axes[1])
axes[1].axvline(avg_psnr, color='red', linestyle='dashed', linewidth=2, label=f'Mean: {avg_psnr:.2f} dB')
axes[1].axvline(30.0, color='green', linestyle='dotted', linewidth=2, label='Mốc xuất sắc (30 dB)')
axes[1].set_title('Phân phối điểm PSNR (Độ nét)', fontsize=12, pad=10)
axes[1].set_xlabel('PSNR (dB)')
axes[1].set_ylabel('Số lượng Batch')
axes[1].legend()

# Subplot 3: Phân phối SSIM
sns.histplot(all_ssim_scores, bins=20, kde=True, color='#9b59b6', ax=axes[2])
axes[2].axvline(avg_ssim, color='red', linestyle='dashed', linewidth=2, label=f'Mean: {avg_ssim:.4f}')
axes[2].axvline(0.90, color='green', linestyle='dotted', linewidth=2, label='Mốc chuẩn (0.90)')
axes[2].set_title('Phân phối điểm SSIM (Cấu trúc)', fontsize=12, pad=10)
axes[2].set_xlabel('Chỉ số SSIM')
axes[2].set_ylabel('Số lượng Batch')
axes[2].legend()

plt.tight_layout()
# Lưu bức ảnh dashboard hoàn chỉnh có đầy đủ cả 3 đồ thị con
fig.savefig('../results/metrics_dashboard.png', dpi=300, bbox_inches='tight')
plt.show()